<a href="https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

## 1. Build the feature vector

Building on the March 2026 slice from ML-04 (data contract). Same grain: one row per
page (`client_hash_id × content_hash_id`), aggregated over the month.

In [2]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions_month,
        SUM(gsc_clicks) AS total_clicks_month,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_month,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS total_sessions_month,
        SUM(scroll_events) AS total_scroll_events_month
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY client_hash_id, content_hash_id
""").df()

# Fills: missing avg_position_month (no valid days) -> 0, meaning "no position data"
features["avg_position_month"] = features["avg_position_month"].fillna(0)
features["ctr_month"] = (features["total_clicks_month"] / features["total_impressions_month"]).fillna(0)

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 9)


,client_hash_id,content_hash_id,total_impressions_month,total_clicks_month,avg_position_month,days_with_impressions,total_sessions_month,total_scroll_events_month,ctr_month
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,5.331238,29,0.0,NaN,0.000000
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,1.0,5.942308,16,0.0,NaN,0.021739
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.908100,31,0.0,NaN,0.001112
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,6.419872,17,0.0,NaN,0.000000
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,6.969536,30,0.0,NaN,0.000000


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `total_impressions_month` | trailing sum of GSC impressions | 0 if no rows | knowable by month end |
| `total_clicks_month` | trailing sum of GSC clicks | 0 if no rows | knowable by month end |
| `avg_position_month` | mean position on days with valid position | filled 0 = "no data" | knowable by month end |
| `days_with_impressions` | count of active search days | 0 if none | knowable by month end |
| `total_sessions_month` | GA4 sessions, filtered on availability flag | 0 if GA4 unavailable | knowable by month end (subject to tracking start) |
| `total_scroll_events_month` | trailing sum of scroll events | 0 if no rows | knowable by month end |
| `ctr_month` | derived: clicks/impressions | 0 if no impressions | knowable by month end |

None of these are categorical — all numeric trailing aggregates. No feature requires
information from after March 31.

In [3]:
print(features.isna().sum())
print("\nZero-impression rows (ctr undefined, filled 0):", (features["total_impressions_month"] == 0).sum())

client_hash_id                   0
content_hash_id                  0
total_impressions_month          0
total_clicks_month               0
avg_position_month               0
days_with_impressions            0
total_sessions_month             0
total_scroll_events_month    70700
ctr_month                        0
dtype: int64

Zero-impression rows (ctr undefined, filled 0): 154699


## 3. The leakage hunt

**Suspect feature:** `second_half_clicks` — used to construct the label itself
(`is_declining` = second-half clicks < first-half clicks). Testing WITH vs WITHOUT.

**Split comparison:** random split vs. grouped-by-client split — does the model's score
collapse once it can't see other pages from the same client during training?

In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit

half = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS first_half_clicks,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS second_half_clicks
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY client_hash_id, content_hash_id
""").df()
half["is_declining"] = (half["second_half_clicks"] < half["first_half_clicks"]).astype(int)

full = features.merge(half, on=["client_hash_id", "content_hash_id"])
y = full["is_declining"]
base_rate = y.mean()
print(f"Base rate (is_declining): {base_rate:.3f}")

honest_cols = ["total_impressions_month", "total_clicks_month", "avg_position_month",
               "days_with_impressions", "total_sessions_month", "total_scroll_events_month", "ctr_month"]

# --- Test 1: WITH vs WITHOUT the suspect feature (random split) ---
X_leak = full[honest_cols + ["second_half_clicks"]].fillna(0)
X_honest = full[honest_cols].fillna(0)

Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leak, y, test_size=0.3, random_state=42, stratify=y)
m_leak = LogisticRegression(max_iter=1000).fit(Xl_tr, yl_tr)
auc_leak = roc_auc_score(yl_te, m_leak.predict_proba(Xl_te)[:, 1])

Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
m_honest = LogisticRegression(max_iter=1000).fit(Xh_tr, yh_tr)
auc_honest = roc_auc_score(yh_te, m_honest.predict_proba(Xh_te)[:, 1])

print(f"\nWITH suspect feature:    ROC-AUC = {auc_leak:.3f}  (base rate {base_rate:.3f})")
print(f"WITHOUT suspect feature: ROC-AUC = {auc_honest:.3f}  (base rate {base_rate:.3f})")
print(f"Collapse: {auc_leak - auc_honest:.3f} — this gap is the leakage confession.")

# --- Test 2: random split vs grouped-by-client split (honest features only) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X_honest, y, groups=full["client_hash_id"]))
m_grouped = LogisticRegression(max_iter=1000).fit(X_honest.iloc[train_idx], y.iloc[train_idx])
auc_grouped = roc_auc_score(y.iloc[test_idx], m_grouped.predict_proba(X_honest.iloc[test_idx])[:, 1])

print(f"\nRandom split (honest features):  ROC-AUC = {auc_honest:.3f}")
print(f"Grouped split (honest features):  ROC-AUC = {auc_grouped:.3f}")
print(f"Gap: {auc_honest - auc_grouped:.3f}")

Base rate (is_declining): 0.087

WITH suspect feature:    ROC-AUC = 1.000  (base rate 0.087)
WITHOUT suspect feature: ROC-AUC = 0.911  (base rate 0.087)
Collapse: 0.089 — this gap is the leakage confession.

Random split (honest features):  ROC-AUC = 0.911
Grouped split (honest features):  ROC-AUC = 0.885
Gap: 0.026


In [5]:
# Deeper check: total_clicks_month literally sums second_half_clicks — a sibling leak
X_stricter = full[["total_impressions_month", "avg_position_month", "days_with_impressions",
                    "total_sessions_month", "total_scroll_events_month"]].fillna(0)
# total_clicks_month and ctr_month dropped too — both derived from/containing second_half_clicks

Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X_stricter, y, test_size=0.3, random_state=42, stratify=y)
m_stricter = LogisticRegression(max_iter=1000).fit(Xs_tr, ys_tr)
auc_stricter = roc_auc_score(ys_te, m_stricter.predict_proba(Xs_te)[:, 1])
print(f"Stricter (no total_clicks_month/ctr_month, both contain the leak): ROC-AUC = {auc_stricter:.3f}")

Stricter (no total_clicks_month/ctr_month, both contain the leak): ROC-AUC = 0.903


**Findings:** Including `second_half_clicks` directly pushed AUC to a perfect 1.000 — a
clean leakage confession (skill: "a feature towers over all others, score near-perfect").
Removing it dropped AUC to 0.911, still high for a base rate of 0.087. On closer
inspection, `total_clicks_month` = `first_half_clicks + second_half_clicks` — a sibling
column that still contains the leaking value as a summand, not just a correlate.
Excluding both `total_clicks_month` and `ctr_month` (derived from it) dropped AUC further
to 0.902 — still elevated, but this is the more honest number of the three, built only
from impressions, position, session, and scroll signals that don't structurally contain
the label.

Grouped-by-client split (on the 0.911 honest-but-not-stricter feature set) dropped AUC
from 0.911 to 0.885 — a 0.026 gap, suggesting mild client-level memorization on the
random split, worth reporting alongside the headline number rather than hiding it.

## 4. What I excluded and why

| Excluded field | Why |
|---|---|
| `fact_content_query_90d` (not joined) | 90-day window overlaps the snapshot's final months; risks pulling post-window information into a March-only slice |
| `client_hash_id`, `content_hash_id` | pseudonymous join/grouping keys only, never features |
| Any FlyRank product decision flag (`health_score`, `priority_score`, `action_type`) | not shipped in this data by design — using a rebuilt version would be a circular result, not discovery |
| `second_half_clicks` | the label's own source column — confirmed as a direct leak (AUC collapsed from 1.000 to 0.911 on removal) |
| `total_clicks_month`, `ctr_month` | sibling columns: `total_clicks_month` literally sums `second_half_clicks`, so it still structurally contains the leak even without naming it directly — confirmed by a further AUC drop to 0.902 |

In [6]:
excluded = ["client_hash_id", "content_hash_id", "second_half_clicks",
            "total_clicks_month", "ctr_month",
            "health_score", "priority_score", "action_type"]
used_stricter = ["total_impressions_month", "avg_position_month", "days_with_impressions",
                  "total_sessions_month", "total_scroll_events_month"]

print("Used (stricter, honest) features:", used_stricter)
print("Explicitly excluded:", excluded)

Used (stricter, honest) features: ['total_impressions_month', 'avg_position_month', 'days_with_impressions', 'total_sessions_month', 'total_scroll_events_month']
Explicitly excluded: ['client_hash_id', 'content_hash_id', 'second_half_clicks', 'total_clicks_month', 'ctr_month', 'health_score', 'priority_score', 'action_type']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.